In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120

df = pd.read_csv('data/final_scored.csv')

# Rename segments to proper business names
SEG_MAP = {
    'Segment A': 'Premium Loyalists',
    'Segment B': 'Churn Risk',
    'Segment C': 'Occasional Shoppers',
    'Segment D': 'High-Value Deal Seekers',
}
df['segment_name'] = df['segment_name'].map(SEG_MAP).fillna(df['segment_name'])

print(f"Loaded: {df.shape}")
print(df['segment_name'].value_counts().to_string())

In [ ]:
# Compute segment-level stats for bubble positions
seg_stats = df.groupby('segment_name').agg(
    median_monetary = ('monetary',    'median'),
    avg_churn_proba = ('churn_proba', 'mean'),
    size            = ('household_key', 'count'),
    churn_rate      = ('churned',      'mean'),
).reset_index()

colors = {
    'Premium Loyalists'      : '#1D9E75',
    'Churn Risk'             : '#D85A30',
    'Occasional Shoppers'    : '#EF9F27',
    'High-Value Deal Seekers': '#7F77DD',
}

fig, ax = plt.subplots(figsize=(10, 7))

for _, row in seg_stats.iterrows():
    c = colors.get(row['segment_name'], 'gray')
    ax.scatter(row['avg_churn_proba'], row['median_monetary'],
               s=row['size'] * 0.4, color=c, alpha=0.75, edgecolors='white', linewidth=1.5)
    ax.annotate(
        f"{row['segment_name']}\n({row['size']} HHs · churn {row['churn_rate']*100:.1f}%)",
        xy=(row['avg_churn_proba'], row['median_monetary']),
        xytext=(12, 0), textcoords='offset points',
        fontsize=9, va='center'
    )

# Quadrant lines at midpoints
mid_x = seg_stats['avg_churn_proba'].mean()
mid_y = seg_stats['median_monetary'].mean()
ax.axvline(mid_x, color='gray', linestyle='--', linewidth=0.8, alpha=0.5)
ax.axhline(mid_y, color='gray', linestyle='--', linewidth=0.8, alpha=0.5)

# Quadrant labels
ax.text(0.01, 4800, '★ Protect & Delight',   fontsize=9, color='#1D9E75', style='italic')
ax.text(0.12, 4800, '⚠ Watch Closely',        fontsize=9, color='#EF9F27', style='italic')
ax.text(0.01, 200,  '✓ Stable — Low Priority', fontsize=9, color='gray',    style='italic')
ax.text(0.12, 200,  '🚨 Immediate Action',     fontsize=9, color='#D85A30', style='italic')

ax.set_xlabel('Average churn probability (model score)', fontsize=11)
ax.set_ylabel('Median household spend ($)', fontsize=11)
ax.set_title('Retention strategy matrix\n(bubble size = segment size)', fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
# Revenue at risk = high-risk households × their median annual spend
high_risk = df[df['churn_risk_tier'] == 'High']

rev_at_risk = (high_risk.groupby('segment_name')
               .agg(hh_count=('household_key', 'count'),
                    total_spend=('monetary', 'sum'),
                    median_spend=('monetary', 'median'))
               .sort_values('total_spend', ascending=False))

rev_at_risk['pct_of_total'] = (rev_at_risk['total_spend'] /
                                   rev_at_risk['total_spend'].sum() * 100).round(1)

print("Revenue at risk (high-risk households only):\n")
print(rev_at_risk.to_string())
print(f"\nTotal revenue at risk : ${rev_at_risk['total_spend'].sum():,.0f}")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))

rev_at_risk['total_spend'].plot(kind='bar', ax=ax1,
    color=[colors.get(s, 'gray') for s in rev_at_risk.index],
    edgecolor='white')
ax1.set_title('Total spend at risk by segment')
ax1.set_ylabel('Total spend ($)')
ax1.tick_params(axis='x', rotation=20)

ax2.pie(rev_at_risk['total_spend'],
       labels=rev_at_risk.index,
       colors=[colors.get(s, 'gray') for s in rev_at_risk.index],
       autopct='%1.1f%%', startangle=90, pctdistance=0.75)
ax2.set_title('Revenue at risk — share by segment')

plt.tight_layout()
plt.show()

In [ ]:
# Priority score = churn_proba × log(monetary)  — ranks by risk AND value
df['priority_score'] = df['churn_proba'] * np.log1p(df['monetary'])

action_list = (df[df['churn_risk_tier'].isin(['High', 'Medium'])]
    [['household_key', 'segment_name', 'monetary', 'frequency',
      'recency', 'churn_proba', 'churn_risk_tier', 'priority_score']]
    .sort_values('priority_score', ascending=False)
    .reset_index(drop=True)
)
action_list.index += 1  # rank from 1

print(f"Total households to action: {len(action_list)}")
print(f"\nTop 20 priority households:\n")
print(action_list.head(20).to_string())

# Save the full action list
action_list.to_csv('data/retention_action_list.csv')
print(f"\n✓ Saved → data/retention_action_list.csv")